# Evaluation and Fairness Reproducibility Notebook

This notebook reproduces the held-out evaluation table, model-comparison figure, and demographic fairness screening table from saved test-set predictions.

Expected input file: `outputs/predictions/heldout_predictions.csv`

Minimum required columns: `target` and one probability column per model. Demographic columns `applicant_sex` and `applicant_age` are used when present.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
from statsmodels.stats.proportion import proportions_ztest

RANDOM_SEED = 42
BOOTSTRAP_ITERATIONS = 1000
THRESHOLD = 0.5
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in {"evaluation", "multilevel", "fairness", "models", "eda"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

PREDICTION_PATHS = [
    PROJECT_ROOT / "outputs" / "predictions" / "heldout_predictions.csv",
    PROJECT_ROOT / "evaluation" / "heldout_predictions.csv",
    PROJECT_ROOT / "outputs" / "heldout_predictions.csv"
]
OUT_DIR = PROJECT_ROOT / "outputs" / "evaluation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

## 1. Load held-out predictions

In [ ]:
def load_predictions(paths):
    for path in paths:
        if path.exists():
            return pd.read_csv(path), path
    raise FileNotFoundError("Place heldout_predictions.csv in outputs/predictions/, evaluation/, or outputs/.")

pred, pred_path = load_predictions(PREDICTION_PATHS)
print(f"Loaded {pred_path}")
print(f"Rows: {len(pred):,}")
if len(pred) != 66460:
    print(f"Warning: expected 66,460 held-out rows from the report, found {len(pred):,} rows.")
pred.head()

In [ ]:
rename_map = {"y_true": "target", "actual": "target", "derived_sex": "applicant_sex", "sex": "applicant_sex"}
pred = pred.rename(columns={k: v for k, v in rename_map.items() if k in pred.columns})
if "target" not in pred.columns:
    raise ValueError("The prediction file must include a target column where 1 = approved and 0 = denied.")
pred["target"] = pd.to_numeric(pred["target"], errors="coerce").astype(int)
pred["denied_true"] = 1 - pred["target"]
pred["denied_true"].value_counts().sort_index()

## 2. Identify probability columns

The notebook prefers denied-probability columns. If only approved-probability columns are present, it converts them to denied probabilities.

In [ ]:
model_column_candidates = {
    "Logistic Regression": ["prob_denied_logistic_regression", "prob_denied_lr", "p_denied_lr", "prob_approved_logistic_regression", "prob_approved_lr", "p_approved_lr"],
    "Decision Tree": ["prob_denied_decision_tree", "prob_denied_dt", "p_denied_dt", "prob_approved_decision_tree", "prob_approved_dt", "p_approved_dt"],
    "Random Forest": ["prob_denied_random_forest", "prob_denied_rf", "p_denied_rf", "prob_approved_random_forest", "prob_approved_rf", "p_approved_rf"],
    "XGBoost": ["prob_denied_xgboost", "prob_denied_xgb", "p_denied_xgb", "prob_approved_xgboost", "prob_approved_xgb", "p_approved_xgb"],
    "Soft Voting Ensemble": ["prob_denied_soft_voting", "prob_denied_soft_voting_ensemble", "prob_denied_voting", "p_denied_voting", "prob_approved_soft_voting", "prob_approved_soft_voting_ensemble", "prob_approved_voting", "p_approved_voting"]
}

probability_columns = {}
for model_name, candidates in model_column_candidates.items():
    found = None
    for candidate in candidates:
        if candidate in pred.columns:
            found = candidate
            break
    if found is not None:
        if "approved" in found:
            converted = "prob_denied_from_" + found
            pred[converted] = 1 - pd.to_numeric(pred[found], errors="coerce")
            probability_columns[model_name] = converted
        else:
            probability_columns[model_name] = found

if not probability_columns:
    raise ValueError("No model probability columns were found. Add predicted denial or approval probabilities for each model.")
probability_columns

## 3. Compute held-out metrics for the denied class

In [ ]:
def compute_metrics(y_true_denied, p_denied, threshold=0.5):
    y_pred_denied = (p_denied >= threshold).astype(int)
    return {
        "PR-AUC": average_precision_score(y_true_denied, p_denied),
        "ROC-AUC": roc_auc_score(y_true_denied, p_denied),
        "F1 (0.5)": f1_score(y_true_denied, y_pred_denied, zero_division=0),
        "Precision": precision_score(y_true_denied, y_pred_denied, zero_division=0),
        "Recall": recall_score(y_true_denied, y_pred_denied, zero_division=0),
        "Accuracy": accuracy_score(y_true_denied, y_pred_denied)
    }

rows = []
y_true_denied = pred["denied_true"].to_numpy()
for model_name, col in probability_columns.items():
    p_denied = pd.to_numeric(pred[col], errors="coerce").to_numpy()
    mask = ~np.isnan(p_denied)
    metrics = compute_metrics(y_true_denied[mask], p_denied[mask], THRESHOLD)
    metrics["Model"] = model_name
    metrics["Probability Column"] = col
    rows.append(metrics)

metric_table = pd.DataFrame(rows)[["Model", "PR-AUC", "ROC-AUC", "F1 (0.5)", "Precision", "Recall", "Accuracy", "Probability Column"]]
metric_table.to_csv(OUT_DIR / "table4_heldout_metrics.csv", index=False)
metric_table

## 4. Bootstrap confidence intervals

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
bootstrap_rows = []
indices = np.arange(len(pred))

for model_name, col in probability_columns.items():
    p_denied_all = pd.to_numeric(pred[col], errors="coerce").to_numpy()
    valid = ~np.isnan(p_denied_all)
    y_valid = y_true_denied[valid]
    p_valid = p_denied_all[valid]
    values = []
    for _ in range(BOOTSTRAP_ITERATIONS):
        sample = rng.choice(np.arange(len(y_valid)), size=len(y_valid), replace=True)
        if len(np.unique(y_valid[sample])) < 2:
            continue
        values.append(compute_metrics(y_valid[sample], p_valid[sample], THRESHOLD))
    boot = pd.DataFrame(values)
    for metric in ["PR-AUC", "ROC-AUC", "F1 (0.5)", "Precision", "Recall", "Accuracy"]:
        bootstrap_rows.append({
            "Model": model_name,
            "Metric": metric,
            "Mean": boot[metric].mean(),
            "CI Low": boot[metric].quantile(0.025),
            "CI High": boot[metric].quantile(0.975),
            "Half Width": (boot[metric].quantile(0.975) - boot[metric].quantile(0.025)) / 2
        })

bootstrap_table = pd.DataFrame(bootstrap_rows)
bootstrap_table.to_csv(OUT_DIR / "table4_bootstrap_intervals.csv", index=False)
bootstrap_table.head(20)

In [ ]:
formatted = metric_table.drop(columns=["Probability Column"]).copy()
for model_name in formatted["Model"]:
    for metric in ["PR-AUC", "ROC-AUC"]:
        half_width = bootstrap_table[(bootstrap_table["Model"] == model_name) & (bootstrap_table["Metric"] == metric)]["Half Width"]
        if len(half_width):
            idx = formatted.index[formatted["Model"] == model_name][0]
            formatted.loc[idx, metric] = f"{metric_table.loc[idx, metric]:.3f} (±{float(half_width.iloc[0]):.3f})"
for metric in ["F1 (0.5)", "Precision", "Recall", "Accuracy"]:
    formatted[metric] = metric_table[metric].map(lambda x: f"{x:.3f}")
formatted.to_csv(OUT_DIR / "table4_formatted_for_report.csv", index=False)
formatted

## 5. Compare against reported Table 4 values

In [ ]:
reported_table4 = pd.DataFrame([
    {"Model": "Logistic Regression", "Reported PR-AUC": 0.721, "Reported ROC-AUC": 0.793, "Reported F1 (0.5)": 0.612, "Reported Precision": 0.648, "Reported Recall": 0.580, "Reported Accuracy": 0.794},
    {"Model": "Decision Tree", "Reported PR-AUC": 0.748, "Reported ROC-AUC": 0.812, "Reported F1 (0.5)": 0.641, "Reported Precision": 0.657, "Reported Recall": 0.626, "Reported Accuracy": 0.801},
    {"Model": "Random Forest", "Reported PR-AUC": 0.871, "Reported ROC-AUC": 0.901, "Reported F1 (0.5)": 0.743, "Reported Precision": 0.781, "Reported Recall": 0.708, "Reported Accuracy": 0.846},
    {"Model": "XGBoost", "Reported PR-AUC": 0.883, "Reported ROC-AUC": 0.912, "Reported F1 (0.5)": 0.756, "Reported Precision": 0.793, "Reported Recall": 0.722, "Reported Accuracy": 0.852},
    {"Model": "Soft Voting Ensemble", "Reported PR-AUC": 0.886, "Reported ROC-AUC": 0.914, "Reported F1 (0.5)": 0.759, "Reported Precision": 0.798, "Reported Recall": 0.724, "Reported Accuracy": 0.854}
])
comparison = reported_table4.merge(metric_table.drop(columns=["Probability Column"]), on="Model", how="left")
for metric in ["PR-AUC", "ROC-AUC", "F1 (0.5)", "Precision", "Recall", "Accuracy"]:
    comparison[f"Delta {metric}"] = comparison[metric] - comparison[f"Reported {metric}"]
comparison.to_csv(OUT_DIR / "table4_reported_vs_reproduced.csv", index=False)
comparison

## 6. Recreate Figure 10

In [ ]:
plot_table = metric_table.sort_values("PR-AUC", ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(plot_table["Model"], plot_table["PR-AUC"])
ax.set_xlabel("Denied-Class PR-AUC")
ax.set_ylabel("Model")
ax.set_title("Model Performance Comparison")
ax.set_xlim(0, max(1.0, plot_table["PR-AUC"].max() + 0.05))
for i, value in enumerate(plot_table["PR-AUC"]):
    ax.text(value + 0.005, i, f"{value:.3f}", va="center")
fig.tight_layout()
fig_path = OUT_DIR / "fig10_model_comparison_recreated.png"
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved {fig_path}")

## 7. Demographic fairness screening and z-tests

In [ ]:
def normalize_sex(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().lower()
    if "joint" in s or "both" in s or "male and female" in s:
        return "Joint"
    if s in {"male", "1"}:
        return "Solo male"
    if s in {"female", "2"}:
        return "Solo female"
    if "female" in s and "male" not in s:
        return "Solo female"
    if "male" in s and "female" not in s:
        return "Solo male"
    return str(value).strip()

def normalize_age(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().lower().replace(" ", "")
    if s in {"<25", "under25", "lessthan25"}:
        return "Under 25"
    if s in {">74", "74+", "over74", "greaterthan74"}:
        return "Over 74"
    return str(value).strip()

def approval_rate(mask):
    subset = pred.loc[mask, "target"]
    return subset.sum(), subset.count(), subset.mean()

def ztest_for_masks(mask_a, mask_b):
    approved_a, n_a, rate_a = approval_rate(mask_a)
    approved_b, n_b, rate_b = approval_rate(mask_b)
    z, p = proportions_ztest([approved_a, approved_b], [n_a, n_b])
    return approved_a, n_a, rate_a, approved_b, n_b, rate_b, z, p

fairness_rows = []
if "applicant_sex" in pred.columns:
    pred["sex_group"] = pred["applicant_sex"].apply(normalize_sex)
    comparisons = [
        ("Joint vs. solo female", pred["sex_group"].eq("Joint"), pred["sex_group"].eq("Solo female")),
        ("Joint vs. solo male", pred["sex_group"].eq("Joint"), pred["sex_group"].eq("Solo male")),
        ("Solo male vs. solo female", pred["sex_group"].eq("Solo male"), pred["sex_group"].eq("Solo female"))
    ]
    for label, mask_a, mask_b in comparisons:
        if mask_a.sum() > 0 and mask_b.sum() > 0:
            approved_a, n_a, rate_a, approved_b, n_b, rate_b, z, p = ztest_for_masks(mask_a, mask_b)
            fairness_rows.append({"Comparison": label, "Group A Approval Rate": rate_a, "Group B Approval Rate": rate_b, "Difference pp": (rate_a - rate_b) * 100, "z": z, "p": p, "Group A n": n_a, "Group B n": n_b, "Flag": "Yes" if abs((rate_a - rate_b) * 100) > 5 else "No"})

if "applicant_age" in pred.columns:
    pred["age_group"] = pred["applicant_age"].apply(normalize_age)
    mask_a = pred["age_group"].eq("Under 25")
    mask_b = pred["age_group"].eq("Over 74")
    if mask_a.sum() > 0 and mask_b.sum() > 0:
        approved_a, n_a, rate_a, approved_b, n_b, rate_b, z, p = ztest_for_masks(mask_a, mask_b)
        fairness_rows.append({"Comparison": "Under 25 vs. over 74", "Group A Approval Rate": rate_a, "Group B Approval Rate": rate_b, "Difference pp": (rate_a - rate_b) * 100, "z": z, "p": p, "Group A n": n_a, "Group B n": n_b, "Flag": "Yes" if abs((rate_a - rate_b) * 100) > 5 else "No"})

fairness_table = pd.DataFrame(fairness_rows)
if len(fairness_table):
    fairness_table.to_csv(OUT_DIR / "table5_fairness_screening_with_ztests.csv", index=False)
    display(fairness_table)
else:
    print("Fairness screening requires applicant_sex and/or applicant_age columns in the prediction file.")

## 8. Equalized-odds screening from denied-class predictions

In [ ]:
soft_col = probability_columns.get("Soft Voting Ensemble")
eqodds_rows = []

if soft_col is not None:
    pred["soft_pred_denied"] = (pd.to_numeric(pred[soft_col], errors="coerce") >= THRESHOLD).astype(int)
    if "sex_group" in pred.columns:
        for group, group_df in pred.dropna(subset=["sex_group"]).groupby("sex_group"):
            y = group_df["denied_true"].to_numpy()
            yhat = group_df["soft_pred_denied"].to_numpy()
            cm = confusion_matrix(y, yhat, labels=[0, 1])
            tn, fp, fn, tp = cm.ravel()
            tpr = tp / (tp + fn) if (tp + fn) else np.nan
            fpr = fp / (fp + tn) if (fp + tn) else np.nan
            eqodds_rows.append({"Group Type": "Applicant sex", "Group": group, "TPR denied": tpr, "FPR denied": fpr, "TN": tn, "FP": fp, "FN": fn, "TP": tp})
    if "age_group" in pred.columns:
        for group, group_df in pred.dropna(subset=["age_group"]).groupby("age_group"):
            y = group_df["denied_true"].to_numpy()
            yhat = group_df["soft_pred_denied"].to_numpy()
            cm = confusion_matrix(y, yhat, labels=[0, 1])
            tn, fp, fn, tp = cm.ravel()
            tpr = tp / (tp + fn) if (tp + fn) else np.nan
            fpr = fp / (fp + tn) if (fp + tn) else np.nan
            eqodds_rows.append({"Group Type": "Applicant age", "Group": group, "TPR denied": tpr, "FPR denied": fpr, "TN": tn, "FP": fp, "FN": fn, "TP": tp})

eqodds_table = pd.DataFrame(eqodds_rows)
if len(eqodds_table):
    eqodds_table.to_csv(OUT_DIR / "equalized_odds_screening_soft_voting.csv", index=False)
    display(eqodds_table)
else:
    print("Equalized-odds screening requires soft-voting predictions and subgroup columns.")